# 04 — Добор недостающих языков (IT, PT, JA, PL)

В Yandex-датасете покрыто 4 языка (RU/ES/EN/FR). ТЗ требует 8 (+ IT/PT/JA/PL). Добираем по 1 треку:
1. yt-dlp скачивает audio из YouTube
2. demucs htdemucs v4 → vocal m4a (та же утилита, что в проде Unbake)
3. LRCLib даёт лирику
4. baseline + alignment как для основного датасета

**Disclaimer:** YouTube как источник — для proof-of-architecture теста. В проде — лицензированные мастера/lossless источники.

Запускать на Colab T4 ПОСЛЕ 03_full_bench (использует ту же машину).

In [ ]:
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))

In [ ]:
# Если репо ещё не склонировано (запуск 04 без 03 в этой сессии)
import os
if not os.path.exists("unbake-test"):
    os.system("git clone -q https://github.com/RezPoint/unbake-test.git")
%cd unbake-test
!pip install -q faster-whisper transformers torchaudio librosa jiwer pydantic requests yt-dlp demucs

In [ ]:
EXTRA = [
    {"lang": "it", "artist": "Maneskin",  "track": "I WANNA BE YOUR SLAVE"},
    {"lang": "pt", "artist": "Anitta",    "track": "Envolver"},
    {"lang": "ja", "artist": "YOASOBI",   "track": "Idol"},
    {"lang": "pl", "artist": "sanah",     "track": "Szampan"},
]

In [ ]:
# 1. Download via yt-dlp (ytsearch1: picks first result, robust to dead IDs)
import subprocess
from pathlib import Path
DOWNLOAD_DIR = Path("data/_yt_raw"); DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
for t in EXTRA:
    safe = f"{t['lang']}_{t['artist']}_{t['track']}".replace(" ", "_")
    out = DOWNLOAD_DIR / f"{safe}.m4a"
    if out.exists():
        print(f"skip {out.name}"); t["raw_path"] = str(out); continue
    query = f"ytsearch1:{t['artist']} {t['track']} official audio"
    cmd = ["yt-dlp", "--no-warnings", "-f", "ba[ext=m4a]/ba", "--no-playlist",
           "-o", str(out), query]
    print("downloading", out.name)
    r = subprocess.run(cmd)
    if r.returncode != 0 or not out.exists():
        print(f"  FAILED ({r.returncode}) — skipping {t['lang']}")
        t["raw_path"] = None
        continue
    t["raw_path"] = str(out)

In [ ]:
# 2. demucs htdemucs v4 -> vocal -> ffmpeg -> m4a 256k
OUT_RAW = Path("data/raw"); OUT_RAW.mkdir(parents=True, exist_ok=True)
DEMUCS_OUT = Path("data/_demucs"); DEMUCS_OUT.mkdir(parents=True, exist_ok=True)
for t in EXTRA:
    if not t.get("raw_path"):
        print(f"skip {t['lang']} (no raw audio)"); continue
    lang_dir = OUT_RAW / t["lang"]; lang_dir.mkdir(parents=True, exist_ok=True)
    final = lang_dir / f"{t['artist']} - {t['track']}.m4a"
    if final.exists():
        print("skip", final.name); t["vocal_path"] = str(final); continue
    print("demucs:", t["raw_path"])
    subprocess.run(["python", "-m", "demucs", "-n", "htdemucs", "--two-stems=vocals",
                    "-o", str(DEMUCS_OUT), t["raw_path"]], check=True)
    base = Path(t["raw_path"]).stem
    src = DEMUCS_OUT / "htdemucs" / base / "vocals.wav"
    subprocess.run(["ffmpeg", "-y", "-i", str(src), "-c:a", "aac", "-b:a", "256k", str(final)], check=True)
    t["vocal_path"] = str(final)
    print("->", final.name)

In [ ]:
# 3. Лирики через LRCLib (с NFC + ASCII-fold fallback)
from src.lyrics import search
import time, unicodedata
for t in EXTRA:
    out = Path("data/lyrics") / t["lang"]; out.mkdir(parents=True, exist_ok=True)
    fname = unicodedata.normalize("NFC", f"{t['artist']} - {t['track']}.txt")
    lyr_path = out / fname
    if lyr_path.exists():
        print(f"skip {lyr_path.name}"); continue
    res = search(t["track"], t["artist"])
    if res is None:
        print(f"MISS {t['artist']} - {t['track']}"); continue
    lyr_path.write_text(res.plain_lyrics, encoding="utf-8")
    print(f"ok  {lyr_path.name} ({len(res.plain_lyrics.split())} words)")
    time.sleep(0.5)

In [ ]:
# 4. batch_runner подхватит новые data/raw/<lang>/ файлы; кэш старых не тронет
!python -m src.eval.batch_runner

In [ ]:
import pandas as pd
df = pd.read_csv("docs/results.csv")
print(df.to_string(index=False))

In [ ]:
!cd data/transcripts && tar czf /content/transcripts.tar.gz *.json
!cp docs/results.csv /content/results.csv
print("downloaded:")
print("  /content/transcripts.tar.gz")
print("  /content/results.csv")